# Packages import

In [2]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
import json
import os

# Ceneo Scraper

 1. Provide url of the project's opinions page

In [6]:
product_code ="39747497"
page = 1
url = f"https://www.ceneo.pl/{product_code}#tab=reviews"
print(url)

https://www.ceneo.pl/39747497#tab=reviews


In [8]:
path_to_driver = "D:\\chromedriver-win64\\chromedriver.exe"
s = Service(path_to_driver) 
driver = webdriver.Chrome(service=s)
driver.get(url)
driver.maximize_window()
driver.find_element(by="xpath", value="//*[@id='js_cookie-consent-general']/div/div[2]/button[1]").click()


2. Send request to provided url

In [9]:
response = requests.get(url)
print(response.status_code)

200


3. Fetch product name

In [10]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [11]:
product_name = page_dom.select_one("h1").get_text()
print(product_name)

Gerlach Patelnia Prestige 28cm


4. Fetch all opinions from the webpage

In [12]:

opinions = page_dom.select("div.user-post.user-post__card.js_product-review")
for opinion in opinions:
    print(opinion.select_one("div.user-post__text").get_text())

 Gerlach Patelnia Prestige 28 cm zbiera pozytywne opinie użytkowników, którzy podkreślają jej solidność oraz jakość wykonania. Wiele osób docenia możliwość pieczenia potraw dzięki metalowej konstrukcji, a także efektywność smażenia, co przekłada się na smaczne potrawy. Choć niektórzy wskazują na potrzebę przyzwyczajenia się do używania stali nierdzewnej w porównaniu z teflonem, ogólne odczucia są bardzo pozytywne. 
Patelnia masywna, solidnie wykonana. Trzeba na początek trochę cierpliwości, aby przestawić się ze zwykłych teflonowych na cało-stalową, ponieważ trochę inaczej się na niej smaży oraz ją myje, ale po kilku razach wydaje się być lepsza od tych teflonowych. Mam nadzieję, że przetrwa długie lata i będzie solidniejsza od teflonowych, kolorowanych z zewnątrz patelni. 
Bardzo polecam tym co cenią jakość. w kuchni rewelacja, mało smażymy aledo zapiekania-rewelacja, cała jest metalowa co spokojnie zdaje egzamin w piekarniku, mniej zmywania, polecam
Świetna do przyrządzania posiłków,

In [13]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinions to extract required data

In [15]:

all_opinions=[]

for opinion in opinions:
    single_opinion={
        "opinion_id":opinion.get('data-entry-id'),
        "author":opinion.select_one(".user-post__author-name").get_text().strip(),
        "recommendation":opinion.select_one(".user-post__author-recomendation").get_text().strip(),
        "score":opinion.select_one(".user-post__score-count").get_text().strip(),
        "content":opinion.select_one(".user-post__text").get_text().strip(),
        "pros":[opinion.get_text().strip() for opinion in opinion.select(".review-feature__item--positive")],
        "cons":[opinion.get_text().strip() for opinion in opinion.select(".review-feature__item--negative")],
        "helpful":opinion.select_one(".vote-yes").get('data-vote'),
        "unhelpful":opinion.select_one(".vote-no").get('data-vote'),
        "publish_date":opinion.select_one(".user-post__published time:nth-child(1)").get('datetime').strip() if opinion.select_one(".user-post__published time:nth-child(1)").get('datetime') else None,
        "purchase_date":opinion.select_one(".user-post__published time:nth-child(1)").get('datetime').strip() if opinion.select_one(".user-post__published time:nth-child(1)").get('datetime') else None,

    }
    print(single_opinion)
    all_opinions.append(single_opinion)


    # print(single_opinion)
    # print(opinion.get('data-entry-id'))
    # print(opinion.select_one(".user-post__author-name").get_text())
    # print(opinion.select_one(".user-post__author-recomendation").get_text())
    # print(opinion.select_one(".user-post__score-count").get_text())
    # print(opinion.select_one(".user-post__text").get_text())
    # print(opinion.select(".review-feature__item--positive"))
    # print(opinion.select(".review-feature__item--negative"))
    # print(opinion.select_one(".vote-yes").get('data-vote'))
    # print(opinion.select_one(".vote-no").get('data-vote'))
    # print(opinion.select_one(".user-post__published time:nth-child(1)").get('datetime'))
    # print(opinion.select_one(".user-post__published time:last-of-type").get('datetime'))


print(all_opinions)

{'opinion_id': '12960738', 'author': 'Norbert', 'recommendation': 'Polecam', 'score': '4,5/5', 'content': 'Patelnia masywna, solidnie wykonana. Trzeba na początek trochę cierpliwości, aby przestawić się ze zwykłych teflonowych na cało-stalową, ponieważ trochę inaczej się na niej smaży oraz ją myje, ale po kilku razach wydaje się być lepsza od tych teflonowych. Mam nadzieję, że przetrwa długie lata i będzie solidniejsza od teflonowych, kolorowanych z zewnątrz patelni.', 'pros': ['funkcjonalność', 'wygląd', 'wytrzymałość'], 'cons': [], 'helpful': '1', 'unhelpful': '0', 'publish_date': '2020-08-16 11:03:22', 'purchase_date': '2020-08-16 11:03:22'}
{'opinion_id': '6536885', 'author': 'Helena', 'recommendation': 'Polecam', 'score': '5/5', 'content': 'Bardzo polecam tym co cenią jakość. w kuchni rewelacja, mało smażymy aledo zapiekania-rewelacja, cała jest metalowa co spokojnie zdaje egzamin w piekarniku, mniej zmywania, polecam', 'pros': ['funkcjonalność', 'wygląd', 'wytrzymałość'], 'cons':

In [19]:
driver.find_element(by="xpath", value="//*[@id='reviews']/div/div[6]/button[4]").click()

ElementClickInterceptedException: Message: element click intercepted: Element <button aria-label="Następna" class="pagination__item pagination__next obfuscated-link js_obfuscated-link" data-hash="LzM5NzQ3NDk3L29waW5pZS0y" type="button"></button> is not clickable at point (1654, 911). Other element would receive the click: <div class="sticky-offer__container js_sticky-offers-container">...</div>
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6d42a37c5+15b35]
	chromedriver!GetHandleVerifier [0x7ff6d42a3830+15ba0]
	chromedriver!(No symbol) [0x7ff6d3fed8dd]
	chromedriver!(No symbol) [0x7ff6d404f175]
	chromedriver!(No symbol) [0x7ff6d404c88a]
	chromedriver!(No symbol) [0x7ff6d4049715]
	chromedriver!(No symbol) [0x7ff6d40485c7]
	chromedriver!(No symbol) [0x7ff6d403b236]
	chromedriver!(No symbol) [0x

In [16]:
next = True if page_dom.select_one("button.pagination__next") else False
if next: page+=1


In [17]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [18]:
with open(f"./opinions/{product_code}.json", "w", encoding="utf-8") as f:
    json.dump(all_opinions,f, indent=4, ensure_ascii=False)